# CCA best - neurips_bmmc

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/cca_optuna/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_cca/optimize_cca_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "cca_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load CCA optimizer: {OPTIMIZER_PATH}")
cca_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = cca_opt
optimizer_spec.loader.exec_module(cca_opt)


In [2]:
DATASET_NAME = 'neurips_bmmc'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'batch'
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(10))
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_cca"

cca_opt.DATASET_NAME = DATASET_NAME
cca_opt.DATA_PATH = DATA_PATH
cca_opt.BATCH_KEY = BATCH_KEY
cca_opt.LABEL_KEY = LABEL_KEY
cca_opt.LAYER = LAYER
cca_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
cca_opt.USE_CASW = "CASW" in SELECTED_METRICS
cca_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
cca_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
cca_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
cca_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
cca_opt.USE_KBET = "KBET" in SELECTED_METRICS
cca_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'neurips_bmmc'
assert BATCH_KEY == 'batch'
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(10))
assert cca_opt.DIMS_USE == 30
assert cca_opt.N_NEIGHBORS == N_NEIGHBORS
assert cca_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                           neurips_bmmc
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                batch
label_key                                            cell_type
metrics      [BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KB...
seeds                           [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
layer                                                   counts
Name: configuration, dtype: object

In [3]:
if not Path(cca_opt.R_SCRIPT_PATH).is_file():
    raise FileNotFoundError(cca_opt.R_SCRIPT_PATH)
if not Path(cca_opt.R_LIBS_PATH).is_dir():
    raise FileNotFoundError(cca_opt.R_LIBS_PATH)

cca_opt.ensure_balanced_clustering_importable()
cca_opt.validate_configuration(list(SELECTED_METRICS))
cca_opt.validate_dataset_contents()


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/pyximport/pyximport.py:51: DeprecationWarning: the imp module is deprecated in favour of importlib and slated for removal in Python 3.12; see the module's documentation for alternative uses
  import imp
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:671: ImportWarning: PyxLoader.exec_module() not found; falling back to load_module()
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group


Dataset preflight: n_obs=90261 n_batches=12 n_labels=45
TRAJ preflight: usable_pseudotime=51407 unique_values=51251
ISOLISI preflight: min_batch_coverage=1 isolated_labels=['CD8+ T naive CD127+ CD26- CD101-', 'cDC1', 'T prog cycling']
CPCR preflight: eligible_labels=40


In [4]:
OPTUNA_STUDY_NAME = 'cca_bmmc_joint_metrics_v4_clean_reduced_repo_metrics'
OPTUNA_SOURCE_JSON = 'results/optuna_multi_metrics/cca/bmmc/cca_bmmc_joint_metrics_v4_clean_reduced_repo_metrics_best_params.json'
OPTUNA_BEST_TRIAL = 8
OPTUNA_BEST_VALUE = 0.6281964349410052
OPTUNA_BEST_PARAMS = {'nfeatures': 2500, 'npcs': 40}

CCA_BEST_ARGS = {
    "nfeatures": OPTUNA_BEST_PARAMS["nfeatures"],
    "npcs": OPTUNA_BEST_PARAMS["npcs"],
    "dims_use": 30,
    "layer": "counts",
}


def run_cca_best(adata, seed):
    return cca_opt.run_cca(
        adata_obj=adata,
        nfeatures=CCA_BEST_ARGS["nfeatures"],
        npcs=CCA_BEST_ARGS["npcs"],
        dims_use=CCA_BEST_ARGS["dims_use"],
        layer=CCA_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "source_json": OPTUNA_SOURCE_JSON,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **CCA_BEST_ARGS,
    },
    name="best_setting",
)


study          cca_bmmc_joint_metrics_v4_clean_reduced_repo_m...
source_json    results/optuna_multi_metrics/cca/bmmc/cca_bmmc...
best_trial                                                     8
best_value                                              0.628196
nfeatures                                                   2500
npcs                                                          40
dims_use                                                      30
layer                                                     counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return cca_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_cca_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()
<frozen importlib._bootstrap>:914: ImportWarning: PyxImporter.find_spec() not found; falling back to find_module()


Cell type silhouette score...


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.6835718777430304


optimised clustering against cca_metric_label
optimal cluster resolution: 0.8
optimal score: 0.689056755791252
BRAS...


seed           9.000000
BAMI           0.689057
CASW           0.589215
SCGRAPHM       0.685939
TRAJ           0.668495
ISOLISI        0.948643
BRAS           0.775359
KBET           0.392709
CPCR           0.769787
bio_score      0.716270
batch_score    0.645952
joint_score    0.688143
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,TRAJ,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
1,1,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
2,2,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
3,3,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
4,4,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
5,5,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
6,6,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
7,7,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
8,8,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143
9,9,0.689057,0.589215,0.685939,0.668495,0.948643,0.775359,0.392709,0.769787,0.71627,0.645952,0.688143


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.689057,1.110223e-16,0.689057,0.689057
CASW,0.589215,0.000000e+00,0.589215,0.589215
SCGRAPHM,0.685939,0.000000e+00,0.685939,0.685939
TRAJ,0.668495,0.000000e+00,0.668495,0.668495
ISOLISI,0.948643,1.110223e-16,0.948643,0.948643
BRAS,0.775359,0.000000e+00,0.775359,0.775359
KBET,0.392709,0.000000e+00,0.392709,0.392709
CPCR,0.769787,1.110223e-16,0.769787,0.769787
bio_score,0.716270,0.000000e+00,0.716270,0.716270
batch_score,0.645952,1.110223e-16,0.645952,0.645952
